# training-step-cycle — ex1: order the five calls of the canonical training step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `training-step-cycle`. Running the final beacon cell reports progress against the `PyTorch: Training step cycle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## The PyTorch training step cycle — quick refresher

Every PyTorch training loop body, no matter how exotic the model, walks the same four-call cycle on each batch:

```
logits = model(x)             # 1. forward
loss   = loss_fn(logits, y)   # 2. loss
loss.backward()               # 3. backward → gradients into .grad
optimizer.step()              # 4. apply update from .grad to params
optimizer.zero_grad()         # 5. clear .grad so next batch starts fresh
```

**Order matters.** `.backward()` must come before `.step()` (no grads → no update). `.zero_grad()` must come after `.step()` (or before the next forward) — otherwise gradients from the previous batch accumulate into the next one. The default is gradient ACCUMULATION; `.zero_grad()` is what makes each batch independent.

### Exercise 1 — order the five calls of the canonical training step

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the canonical 5-call training-step cycle (forward → loss → backward → step → zero_grad) in the correct order against a 1-parameter model so that each step strictly decreases the loss.
> Keywords: training-loop, step-order, mini-batch-sgd
> ```

**KCs targeted:** `training-step-five-call-order`, `training-step-zero-grad-resets-accumulation`

Implement `ex1_train_one_param(w_init, x, y, lr, n_steps)`. Goal: fit a scalar weight `w` so that `w * x ≈ y` using vanilla SGD via the official `torch.optim.SGD` API. Concretely:

1. Build `w = t.tensor([w_init], requires_grad=True)` — a 1-element leaf with grad tracking.
2. Build `optimizer = t.optim.SGD([w], lr=lr)`.
3. Loop `n_steps` times. In each iteration do the 5 canonical calls IN ORDER:
   - `pred = w * x`           (forward)
   - `loss = ((pred - y) ** 2).mean()`  (loss)
   - `loss.backward()`        (backward)
   - `optimizer.step()`       (step)
   - `optimizer.zero_grad()`  (zero_grad)
4. Record `loss.item()` BEFORE `.backward()` so the test can verify the loss strictly decreases.
5. Return `(w.detach().clone(), losses_list)`.

Inputs:
- `w_init`: float — starting weight value.
- `x, y`: 1-D float tensors of equal length.
- `lr`: float learning rate.
- `n_steps`: int.

Output: tuple `(w_final, losses)`. `w_final` is a detached 1-element tensor. `losses` is a list of `n_steps` Python floats.

In [ ]:
def ex1_train_one_param(w_init: float, x: Tensor, y: Tensor,
                        lr: float, n_steps: int) -> tuple:
    """Fit w so that w*x ~= y. Returns (w_final, losses)."""
    raise NotImplementedError()


def _test_ex1():
    x = t.tensor([1.0, 2.0, 3.0, 4.0])
    y = t.tensor([2.0, 4.0, 6.0, 8.0])   # true w = 2.0
    w_final, losses = ex1_train_one_param(w_init=0.0, x=x, y=y, lr=0.05, n_steps=20)
    assert isinstance(losses, list), f'losses must be a list, got {type(losses)}'
    assert len(losses) == 20, f'expected 20 losses, got {len(losses)}'
    for i, lv in enumerate(losses):
        assert isinstance(lv, float), f'losses[{i}] is {type(lv)}, must be float'
    # Monotonic decrease — proves the 5-call order is correct.
    # If zero_grad is missing or in the wrong spot, gradients
    # accumulate and loss diverges or oscillates.
    for i in range(1, len(losses)):
        assert losses[i] <= losses[i - 1] + 1e-7, (
            f'loss not decreasing at step {i}: {losses[i-1]:.6f} -> {losses[i]:.6f}; '
            f'check call order (likely zero_grad missing or backward-after-step)'
        )
    # After 20 steps with lr=0.05, w should be close to 2.0.
    assert abs(w_final.item() - 2.0) < 0.05, (
        f'expected w ~ 2.0, got {w_final.item():.4f}; '
        f'either step/backward order wrong or gradients not zeroed'
    )
    # First loss must equal mean((0*x - y)^2) = mean(y^2) = 30.0 exactly.
    assert abs(losses[0] - 30.0) < 1e-4, (
        f'first loss should be 30.0 (mean of y^2 since w_init=0), got {losses[0]:.4f}; '
        f'are you appending loss BEFORE backward()?'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_train_one_param(w_init, x, y, lr, n_steps):
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        losses.append(loss.item())   # snapshot BEFORE backward
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return w.detach().clone(), losses
```

**Why snapshot the loss BEFORE backward.** `.item()` on the loss is read-only — its value is fixed at that point. Putting it after `.step()` would show the loss for the CURRENT weights AFTER the update, which still trends down but is a different (and confusing) curve.

**The five calls form a single conceptual unit.** Mentally treat `forward / loss / backward / step / zero_grad` as one indivisible block. Every PyTorch training loop you'll ever write — Karpathy's, ARENA's, HuggingFace's — has this skeleton.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()